In [ ]:
!pip install -q -U google-genai > /dev/null 2>&1

import time
from datetime import datetime, timedelta, timezone
from google import genai
from google.colab import userdata

GOOGLE_API_KEY = userdata.get('GEMINI_API_KEY')
if GOOGLE_API_KEY:
    GOOGLE_API_KEY = GOOGLE_API_KEY.strip()

client = genai.Client(api_key=GOOGLE_API_KEY)
IST = timezone(timedelta(hours=5, minutes=30))
MODELS = ['gemini-3.5-flash']

print("=== 🟢 [SYSTEM ACTIVE] ⚡ SMART GRID HEATWAVE CONTROLLER ===")

while True:
    sensor_data = input("\nEnter Your Latest Wokwi Status Report (or type 'exit'): ")

    if sensor_data.strip().lower() == 'exit':
        print("\nSystem stopped.")
        break

    timestamp = datetime.now(IST).strftime("%d-%b-%Y %I:%M:%S %p IST")

    prompt = f"""
You are a Smart City Control Room Assistant in India.
Read this sensor report: {sensor_data}

Rules:
- Use very simple, clear Indian English so even a beginner can understand.
- Be practical and detailed, but keep it neatly organized with clear spacing.
- Follow these STRICT Indian Smart City Thresholds for your verdict:
  * TEMPERATURE: Below 35°C = Safe | 35°C to 41.9°C = Warning (High Heat) | 42°C and above = Critical (Heatwave Danger).
  * WATER TANK: Above 40% = Safe | 21% to 40% = Warning (Low) | 20% and below = Critical (Severe Shortage).
  * OVERALL ALERT LEVEL:
    - 🔴 CRITICAL if either Temperature or Water is Critical.
    - 🟡 WARNING if either is Warning (and neither is Critical).
    - 🟢 SAFE only if both are Safe.
- Follow this exact format:

========================================
   SMART GRID CONTROL ROOM REPORT
========================================
DATE & TIME    : {timestamp}
ALERT LEVEL    : [🟢 SAFE / 🟡 WARNING / 🔴 CRITICAL]
EMERGENCY TYPE : [None / Heat Emergency / Water Emergency / Both Heat & Water Emergency]

--- 1. SENSOR STATUS ---
- Temperature : [State the temp and whether it is Safe, Warning, or Critical]
- Water Tank  : [State the % and whether it is Safe, Warning, or Critical]
- Cooling Fan : [State ON or OFF, and if it should stay that way or change]

--- 2. SITUATION SUMMARY ---
[Explain in 2-3 clear sentences what this means for the city's power grid and water supply right now.]

--- 3. OPERATOR ACTIONS ---
1. [Power/Equipment Action - clear, practical step]
2. [Water Supply Action - clear, practical step]
3. [Safety/Check Action - clear, practical step]

--- 4. CITIZEN ALERT ---
"[Write 1 short, polite alert message to send to city residents.]"

"""

    success = False
    for model_name in MODELS:
        try:
            response = client.models.generate_content(
                model=model_name,
                contents=prompt
            )
            print("\n" + response.text + "\n")
            with open("incident_log.txt", "a", encoding="utf-8") as log_file:
                log_file.write(response.text + "\n\n")
            print("💾 Report saved to incident_log.txt\n")
            success = True
            break
        except Exception:
            continue


    if not success:
        print("\n⚠️ AI servers are temporarily busy. Please try again in a few seconds.\n")

=== 🟢 [SYSTEM ACTIVE] ⚡ SMART GRID HEATWAVE CONTROLLER ===

   SMART GRID CONTROL ROOM REPORT
DATE & TIME    : 22-Nov-2026 07:17:05 PM IST
ALERT LEVEL    : 🟢 SAFE
EMERGENCY TYPE : None

--- 1. SENSOR STATUS ---
- Temperature : 19.40 °C - Safe (Well below the 35°C limit)
- Water Tank  : 59% - Safe (Well above the 40% limit)
- Cooling Fan : OFF (Should stay OFF as the current temperature is cool and normal)

--- 2. SITUATION SUMMARY ---
All city utility systems are operating smoothly under normal conditions. The equipment temperature is cool at 19.40 °C, and the main water tank has sufficient water at 59%. There is no load or pressure on the power grid and water supply systems right now.

--- 3. OPERATOR ACTIONS ---
1. Keep the cooling fan OFF to save power, and continue monitoring the temperature display.
2. Maintain standard water distribution without any flow restrictions since water levels are fine.
3. Perform routine hourly checks on sensor meters to ensure smooth ongoing operations